# 第10回 主成分分析の解析例


---


## 前回からの接続


前回は、400枚の顔をPCAで縮めました。主成分（principal component）は固有顔として絵で眺められ、50個で持てば十分だとわかりました。

でも、表のデータの主成分は、絵になりません。車の性能表をPCAにかけて、第1主成分が出てきたとして、それは何を表すのでしょうか。今日は、主成分に名前をつけて読む方法を学びます。


---


## 今日の分析目標


あなたは自動車雑誌の編集者です。392台の車を、6つの性能から1枚の地図に並べ、読者に「この車は、どんな車か」を伝えたい。

6つの性能を2本の軸に束ねて地図にし、その軸に名前をつけ、地図の上の位置で1台の車を説明します。もう1つのねらいは、関数に分かれた他人のスクリプトを、関数ごとに読めるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは自動車雑誌の編集者です。誌面で、いろいろな車を読者に紹介します。性能表には、燃費・馬力・重さなどの数が並びます。

決めたいのは、392台の車を、6つの性能から1枚の地図に並べることです。そして読者に、地図の上の位置で「この車は、どんな車か」を伝えたいのです。


### 手元のデータ

seabornに入っている車の性能表`mpg`です。1行が1台で、燃費・気筒数・排気量・馬力・重さ・加速の6つの性能に、製造国・年式・車名が並びます。欠けた値のある行を落とすと、392台です。答えの列はありません。


### 最後に出す答え

2本の軸の名前（右に行くほど、上に行くほど、どんな車か）と、地図の上の位置から1台の車を説明する一文です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 関数に分かれたスクリプトを、関数ごとの役割・入力・出力から読み、使える
- 負荷量（loading）から、主成分に名前をつけられる
- バイプロット（biplot）で、点（車）と矢印（性能）を重ねて読み、矢印の角度と相関を照らし合わせられる
- 累積寄与率（cumulative proportion of variance explained）から、何本の軸を残すかを決められる
- 地図の上の位置で、1台の車の性格を説明できる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出す
- 入力：なし
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ③ の解釈が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (3,)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 配られたスクリプトを読む


軸に名前をつける前に、まず2本の軸を出しておきます。今日は、同僚が書いてくれた完成品のスクリプトから始めます。3つの関数に分かれています。いきなり1行ずつ読むのではなく、まず関数ごとに「何をするか（役割）・何を渡すか（入力）・何が返るか（出力）」をつかみます。


### おさらい：def

```python
def 名前(引数):
    return 戻り値
```

`def`の後ろが関数の名前と引数（入力）、`return`の後ろが戻り値（出力）でした（第3回）。第8回では、自分のコードを関数に分けて1本にまとめました。


### お手本：3つの関数を定義する

- 役割：車の表を読み込む・PCAをかける・地図を描く、の3つの関数を定義する
- 入力：なし（定義するだけ）
- 出力：なし（関数ができる）

`def`のすぐ下の`"""…"""`はドキュメント文字列（docstring）で、その関数の説明です。ここを読めば、中身を読まなくても役割がわかります。


In [ ]:
def load_cars():
    """車の性能表を読み込み、欠けた値のある行を落とす"""
    df = sns.load_dataset('mpg').dropna().reset_index(drop=True)
    return df

def fit_pca(df, cols, k=2):
    """cols を標準化して PCA をかけ、PCA と得点を返す"""
    X_s = StandardScaler().fit_transform(df[cols])
    pca = PCA(n_components=k).fit(X_s)
    return pca, pca.transform(X_s)

def biplot(ax, scores, pca, cols, groups):
    """得点を点で、負荷量を矢印で、1枚の地図に重ねて描く"""
    for g in np.unique(groups):              # グループごとに点
        m = groups == g
        ax.scatter(scores[m, 0], scores[m, 1], s=18, alpha=0.5,
                   label=g)
    L = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
    for (x, y), name in zip(L * 3, cols):    # 負荷量を3倍の矢印に
        ax.arrow(0, 0, x, y, head_width=0.08, color='black')  # 矢印
        ax.text(x * 1.15, y * 1.15, name, ha='center')
    ax.axhline(0, color='gray', lw=0.5)               # 原点を通る線
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_aspect('equal')                  # 角度をゆがめない
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend()


3つの関数を、役割・入力・出力で読むと、こうなります。

| 関数 | 役割 | 入力 | 出力 |
|---|---|---|---|
| `load_cars()` | 車の表を読み込む | なし | 表`df` |
| `fit_pca(df, cols, k)` | 標準化（standardization）してPCA | 表、使う列、軸の数 | PCAと、各車の得点 |
| `biplot(ax, scores, pca, cols, groups)` | 地図を描く | 図の枠、得点、PCA、列、色分け | なし（図を描く） |

`biplot`の中身（負荷量を計算して、矢印の長さを3倍にする、など）は、今は読まなくてかまいません。使うのに必要なのは、入力と出力です。


### 変えてみる：関数を呼んで、2本の軸に縮める

使う列は、燃費（`mpg`）・気筒数（`cylinders`）・排気量（`displacement`）・馬力（`horsepower`）・重さ（`weight`、ポンド）・加速（`acceleration`、0→60マイルの秒数）の6つです。


In [ ]:
df = load_cars()
feat = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
        'acceleration']
pca2, S = fit_pca(df, feat, k=2)
print(len(df), S.shape)
print(pca2.explained_variance_ratio_.round(3))


392台、得点は392行×2列。第1主成分（PC1）の寄与率（proportion of variance explained）は79.8%、第2主成分（PC2）は12.1%。2本の軸で、6つの性能の91.9%を保てます。


---


## 2. 負荷量で、軸に名前をつける


2本の軸で9割を保てました。では、その軸は何を表しているでしょうか。主成分は、6つの列に係数をかけて足したものです。この係数は`pca2.components_`に入っています。行が主成分、列が元の6つの列です。

ただ、係数は「どの向きか」を表す数で、そのままでは列と主成分の関係の強さとして読みにくい。そこで、係数を負荷量に直してから、大きい列（プラスでもマイナスでも）を、その主成分の中身として読みます。


### おさらい：転置

```python
Xc.T          # 行と列を入れかえる（400×4096 → 4096×400）
```

`.T`は、行と列を入れかえる転置でした（第9回）。


### お手本：係数の表を作る

- 役割：PC1・PC2をつくる、6つの列の係数を表にする
- 入力：`pca2`、`feat`
- 出力：6行×2列の表

`.T`で転置して、列ごとに1行にします。


In [ ]:
coef = pd.DataFrame(pca2.components_.T, index=feat,
                    columns=['PC1', 'PC2'])
print(coef.round(2))


係数は、主成分ごとに「2乗して足すと1」になるようにそろえてあります。向きだけを表すので、寄与率の大きいPC1も、小さいPC2も、同じくらいの大きさの数が並びます。


### 変えてみる：負荷量の表にする

- 役割：係数を、列と主成分の関係の強さ（負荷量）に直す
- 入力：係数の表`coef`、`pca2`
- 出力：6行×2列の負荷量の表

`pca2.explained_variance_`は、PC1・PC2の得点の分散です。その平方根（得点の標準偏差）を係数にかけたものが負荷量で、その列と主成分の得点との相関になります（−1〜1）。

`表 * 並び`は、表の1列目に並びの1つ目を、2列目に2つ目を、かけます。


In [ ]:
load = coef * np.sqrt(pca2.explained_variance_)
print(load.round(2))


PC1では、気筒数・排気量・馬力・重さがそろってプラス（0.94〜0.97）、燃費がマイナス（−0.87）です。右に行くほど「大きく、重く、馬力があり、燃費が悪い」車です。PC1は大きさ・パワーの軸と呼べそうです。


### TODO①：PC2と関係の強い列を、大きい順に並べる

`load['PC2']`を、絶対値の大きい順に並べて表示してください。PC2には、何という名前をつけられますか。


In [ ]:
# TODO: PC2 の負荷量を、絶対値の大きい順に並べて表示する
# ヒント: load['PC2'].abs().sort_values(ascending=False)
...


加速（`acceleration`）が0.76と、とびぬけて大きい値です（ほかは0.24以下）。`acceleration`は「0→60マイルにかかる秒数」なので、大きいほどゆっくり加速する車です。PC2は加速のゆるやかさの軸と呼べます。

ただし、この名前は負荷量を見てつけたあだ名です。主成分は、どちら向きを正にするかが計算しだいで反転しうるので（第9回）、反転すれば「加速のすばやさの軸」と呼んでも同じことです。


---


## 3. バイプロット：点と矢印を重ねる


2本の軸に名前がつきました。では、車はその地図のどこにいるでしょうか。各車の得点を点で、6つの列の負荷量を原点からの矢印で、同じ地図に重ねた図をバイプロットと呼びます。点がどちらの矢印の向きにあるかで、その車の性格が読めます。


### お手本：製造国で色分けしたバイプロット

- 役割：1節の関数`biplot`を呼び、製造国（アメリカ・日本・ヨーロッパ）で色分けした地図を描く
- 入力：得点`S`、`pca2`、`feat`、製造国の列
- 出力：バイプロット


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
biplot(ax, S, pca2, feat, df['origin'].values)
plt.show()


アメリカ車（usa）は右に広がり、日本車とヨーロッパ車は左に固まっています。右を向く矢印（気筒数・排気量・馬力・重さ）の側がアメリカ車、左を向く燃費の矢印の側が日本車・ヨーロッパ車です。当時のアメリカ車は、大きく重く、燃費の悪い車が多かったのです。

負荷量で描いた矢印どうしの角度からは、列どうしの関係が読めます。同じ向きなら正の相関、反対向きなら負の相関、直角ならほぼ無相関、が目安です。


### TODO②：相関を計算して、矢印と比べる

「重さと排気量」「燃費と重さ」の相関を計算して、矢印の向き（同じ向き／反対向き）と合っているか確かめてください。


In [ ]:
# TODO: weight と displacement、mpg と weight の相関を表示する
# ヒント: df['weight'].corr(df['displacement'])
...


重さと排気量は0.93（矢印は同じ向き）、燃費と重さは −0.83（矢印は反対向き）。矢印の角度と相関が合っています。


### 変えてみる：燃費と加速

燃費と加速の相関も見てみます。


In [ ]:
print(round(df['mpg'].corr(df['acceleration']), 2))


相関は ＋0.42（正）です。地図では、燃費の矢印は左下、加速の矢印は左上を向いていて、そのあいだの角度は約60度。90度より小さいので、正の相関と合っています。


### 変えてみる：矢印の長さ

- 役割：6つの列が、2本の軸でどれだけ保てているかを並べる
- 入力：負荷量の表`load`
- 出力：列ごとの、保てている割合

`load ** 2`で表のすべての値を2乗し、`.sum(axis=1)`で行ごとに（PC1とPC2を）足します。


In [ ]:
print((load ** 2).sum(axis=1).round(2))


負荷量の2乗を足すと、2本の軸でその列の散らばりを何割保てているかになります。矢印の長さは、その平方根です（この図では3倍してあるので、全部保てていれば長さ3）。

燃費は0.81、加速は0.99、ほかも0.9を超えます。どの矢印も長いので、角度を相関の目安として読めます。保てている割合が小さい列（短い矢印）があれば、その列の角度は当てになりません。2本の軸の地図は、近似です。


---


## 4. 何本の軸を残すか


2本の軸の地図は、近似でした。その2本で91.9%を保てました。では、3本、4本と増やすと、どうなるでしょうか。


### お手本：6本すべての累積寄与率

- 役割：1節の`fit_pca`を`k=6`で呼び、累積寄与率を並べる
- 入力：`df`、`feat`、k＝6
- 出力：6つの累積寄与率


In [ ]:
pca6, _ = fit_pca(df, feat, k=6)        # 得点は使わない
print(np.cumsum(pca6.explained_variance_ratio_).round(3))


1本で79.8%、2本で91.9%、3本で96.3%、4本で98.3%。2本目までで大きく増え、3本目からは、ほとんど増えません。

`_`（アンダースコア）は、「返ってくるけれど使わない値」を受けるときの名前です。`fit_pca`はPCAと得点の2つを返しますが、ここでは得点を使いません。


In [ ]:
# @title 図：スクリープロット（寄与率）
r = pca6.explained_variance_ratio_
plt.bar(range(1, 7), r, color='#0066cc')
plt.plot(range(1, 7), np.cumsum(r), 'o-', color='#e63946', label='累積')
plt.xlabel('主成分'); plt.ylabel('寄与率'); plt.legend()
plt.show()


寄与率を棒で並べた図をスクリープロット（scree plot）と呼びます。棒が急に低くなり、そのあと平らに続きます。平らになる手前の「肘」までを残すのが目安です。この車のデータでは、PC3から先が平らなので、肘までの2本。累積でも91.9%を保てるので、読者に見せる地図は2本で十分です。


---


## 5. 地図の上で、1台を説明する


最初の問いに戻ります。2本の軸に名前がつきました。

- PC1（79.8%）：右ほど「大きく、重く、馬力があり、燃費が悪い」。大きさ・パワーの軸
- PC2（12.1%）：上ほど「ゆっくり加速する」。加速のゆるやかさの軸


### お手本：2台の車を、地図の位置で説明する

- 役割：各車の得点を表に加え、2台を選んで、位置と性能を並べる
- 入力：`df`、得点`S`
- 出力：2台の表

`assign`は、列を加えた表を返すメソッドです（第7回）。`isin`は、「並びのどれかに入っているか」のTrue/Falseを返します。`drop_duplicates('name')`で、同じ名前の車が何年ぶんもあるときに1台に絞ります。


In [ ]:
cars = df.assign(PC1=S[:, 0], PC2=S[:, 1])
two = ['toyota corolla 1200', 'chevrolet impala']
pick = cars[cars['name'].isin(two)].drop_duplicates('name')
cols = ['name', 'origin', 'PC1', 'PC2', 'mpg', 'weight', 'horsepower']
print(pick[cols].round(2))


トヨタ・カローラ1200は、PC1が −2.71（左）。軽く（1773ポンド）、燃費のよい（31マイル／ガロン）小型車です。シボレー・インパラは、PC1が ＋4.92（右）。重く（4354ポンド）、馬力のある（220馬力）大型車です。地図の左右の位置が、そのまま車の性格を表しています。


### TODO③：ホンダ・シビックを探す

名前に`'honda civic'`を含む車をすべて探し、年式（`model_year`）と、PC1・PC2、燃費を表示してください。シビックは、地図のどこにいますか。

`cars['name'].str.contains('toyota')`は、名前に`'toyota'`を含むかのTrue/Falseを返します。`cars[ ]`にTrue/Falseの並びを入れると、Trueの行だけが残ります。


In [ ]:
# TODO: 名前に 'honda civic' を含む車を探して表示する
# ヒント: civic = cars[cars['name'].str.contains('honda civic')]
#         civic[['name', 'model_year', 'PC1', 'PC2', 'mpg']]
...


シビックは8台。1974年の1台（PC1 −0.99）を除き、PC1は −2.2〜−2.7と、地図の左端近くに並びます。燃費は24〜44.6マイル／ガロンで、いちばんよいのは1980年の1500 GL（44.6）。1982年の2台は38と32です。1500 GLはPC2が −1.68と下（きびきび加速する）にいます。


### TODO④：地図のいちばん上にいる車

PC2がいちばん大きい車を探し、名前・製造国・年式・PC1・PC2・加速を表示してください。その車を、読者に一文で紹介してみましょう。

`i = cars['PC1'].idxmax()`は、PC1がいちばん大きい値がある行の番号を返します。`cars.loc[i]`で、その番号の行を1行まるごと取り出せます。


In [ ]:
# TODO: PC2 がいちばん大きい車を探して表示する
# ヒント: cars.loc[cars['PC2'].idxmax()]
...


1979年のプジョー504（ヨーロッパ）です。PC2は2.91、0→60マイルに24.8秒かかる、ゆったり加速する車です。PC1は −2.04と左寄りで、小さめで燃費もよい（27.2マイル／ガロン）。

> 答え：392台の車は、「大きさ・パワー」と「加速のゆるやかさ」の2本の軸で、6つの性能の散らばりの9割を保った地図に並べられる。右のインパラは大きく重いアメリカ車、左のカローラは軽く燃費のよい日本車、上のプジョー504はゆったり加速する車、と地図の位置で説明できる。

まだ言えないことが、2つ残っています。

- 似た車の群れ：地図の上で、似た車どうしをグループに分けるには
- 近くの車を優先した地図：PCAは、まっすぐな向きで全体の散らばりを地図にする。近くの車どうしの関係を優先して地図にすると、何が見えるか（発展）


---


## 目標に答えられたか


- 今日の目標は「392台の車を1枚の地図に並べ、この車はどんな車かを伝える」ことでした。5節で、2本の軸にどんな名前をつけ、2台の車をどう説明しましたか？
- 1節で、3つの関数の役割・入力・出力は何でしたか？`biplot`の中身を読まずに使えたのは、なぜですか？
- 3節のバイプロットで、アメリカ車と日本・ヨーロッパ車は、どちら側に分かれましたか？
- 3節で、矢印の長さから何がわかりましたか？ 角度を相関の目安として読めるのは、どんなときですか？


---


## 今日の要点


- 主成分は各列に係数をかけて足したもの。係数に得点の標準偏差（standard deviation）をかけた負荷量は、列と主成分の相関
- 負荷量の大きい列から、主成分に名前（あだ名）をつける
- バイプロットは、得点（点）と負荷量（矢印）を重ねた地図。点の位置で、その行の性格を読む
- 矢印の角度は相関の目安。矢印の長さは、2本の軸でその列を保てている度合い（短い矢印の角度は当てにならない）
- 累積寄与率とスクリープロットで、残す軸の数を決める（肘まで）
- Python：配られたスクリプトは、関数ごとの役割・入力・出力（docstring、引数、return）から読む。使うのに中身の全部は要らない


---


## 次回へ


今日は、392台の車を地図の上に置いて、1台ずつ説明しました。

次回は、地図の上の点を、似たものどうしのグループに分けます。ラベルがないまま、データからグループを見つけます。142か国を、平均寿命と豊かさから、支援の方針が近いグループに分けられるでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

1節の`fit_pca`は、PCAの前に標準化していました。今度は標準化をしません。元の値`df[feat]`をそのまま`PCA(n_components=2)`に学ばせ、PC1・PC2の寄与率と、2節と同じ係数の表を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`pca_raw = PCA(n_components=2).fit(df[feat])`とします。
寄与率は`.explained_variance_ratio_`、係数の表は2節と同じく`pd.DataFrame(pca_raw.components_.T, index=feat, columns=['PC1', 'PC2'])`です。

</details>


### 応用②（判断）

応用①の、標準化しないPC1の寄与率は何%ですか。%で小数第1位まで答えてください（例: 45.6）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`explained_variance_ratio_[0]`を100倍して、`round(値, 1)`です。

</details>


### 応用③（解釈）

標準化をやめると、PC1の中身は「大きさ・パワーの軸」から大きく変わりました。応用①で係数の絶対値がいちばん大きい列と、その列の単位と値の大きさ（`df[feat].std()`で確かめる）に触れて、「標準化は面倒だから省きたい」と言う同級生に向けて、3行程度で説明してください。


（ここに3行程度で書く）


---


## 発展（任意）


### t-SNEとUMAP：曲がった構造の地図

PCAは、データのまっすぐな向きを取り出して、全体の散らばりを地図にしました。t-SNEとUMAPは、「近くにいた点どうしは、地図でも近くに置く」ことを優先する方法です。似た車どうしの群れが、島のように強調されます。同じ車のデータで、3つの地図を並べてみます。


In [ ]:
import importlib.util
if importlib.util.find_spec('umap') is None:   # 無ければ入れる
    %pip install -q umap-learn
import warnings
import umap
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore', message='n_jobs value')

X_s = StandardScaler().fit_transform(df[feat])
maps = {'PCA': S,
        't-SNE': TSNE(n_components=2, random_state=0).fit_transform(X_s),
        'UMAP': umap.UMAP(random_state=0).fit_transform(X_s)}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, (name, Z) in zip(axes, maps.items()):
    for g in np.unique(df['origin']):
        m = df['origin'].values == g
        ax.scatter(Z[m, 0], Z[m, 1], s=10, alpha=0.6, label=g)
    ax.set_title(name)
axes[0].legend(); plt.tight_layout(); plt.show()


読み方　PCAの地図は、アメリカ車が右、日本・ヨーロッパ車が左に、なめらかに広がっていました。t-SNEとUMAPでは、点が大きな3つの島に分かれます。気筒数は4・6・8のとびとびの値しか（ほぼ）とらないので、同じ気筒数の車どうしが近くに集まり、その群れが島として強調されるのです（下のt-SNEの図は、気筒数で色分けしてあります。PCAの地図を気筒数で色分けしても、気筒数ごとの帯が見えます）。

ただし、t-SNEとUMAPの地図の軸には意味がありません。負荷量もなく、「右ほど大きい車」とは読めません。そして、次の2つに注意が要ります。


In [ ]:
# 注意①：乱数の種を変えると、地図の形が変わる
# init='random' は、はじめの点の置き方をでたらめにする指定
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, seed in zip(axes, [0, 1, 2]):
    tsne = TSNE(n_components=2, init='random', random_state=seed)
    Z = tsne.fit_transform(X_s)
    sc = ax.scatter(Z[:, 0], Z[:, 1], s=10, c=df['cylinders'],
                    cmap='viridis')
    ax.set_title(f't-SNE（種 {seed}）')
fig.colorbar(sc, ax=axes, label='気筒数')     # 色と気筒数の対応
plt.show()


In [ ]:
# 注意②：遠い点どうしの距離は、保たれない
from scipy.spatial.distance import pdist

d_orig = pdist(X_s)                        # 6つの性能での、車どうしの距離
for name, Z in maps.items():
    r = np.corrcoef(d_orig, pdist(Z))[0, 1]
    print(f'{name:6s}: 元の距離との相関 {r:.2f}')


読み方　種を変えると、島の位置や向きがまるで変わります（色は気筒数）。島の中の顔ぶれはだいたい同じですが、「島Aと島Bが近い」といった読み方はできません。

元の距離との相関は、PCAがいちばん高く、t-SNEとUMAPは下がります。t-SNEとUMAPは近くの点の関係を守るかわりに、遠い点どうしの距離を犠牲にしているのです。

t-SNEとUMAPは、「似たものの島があるか」を眺めるのには強い道具ですが、地図の距離や軸を読むならPCA、と使い分けましょう。

試すなら、`umap.UMAP(n_neighbors=5, …)`や`n_neighbors=50`のように、近くとみなす点の数を変えて、島の分かれ方がどう変わるか見てみましょう。
